# L34 · 声音的幻觉 — Nyquist 定理与混叠

**今日目标**：理解为什么采样率必须 > 2×最高频率。亲眼看到一个高频正弦被欠采样后“伪装”成低频。

**为什么对Aurora重要**：Aurora 的 16 kHz 音频管道只处理 0–8 kHz 的频率分量；录音前的抗混叠滤波器把 8 kHz 以上截断，正是为了防止高频折叠成虚假的低频特征进入 mel 频谱或 MFCC。

## 本课剧情：识破高频的伪装

采样太慢时，高频会折叠成一个假的低频——两组完全不同的波，在离散采样点上却完全重合，无法区分。本节用 8 kHz 采 6 kHz 信号来演示这个折叠：`alias = |f - sr·round(f/sr)|` 把真实频率映射成 0 到 Nyquist 之间的视在频率。

## 1. 概念

采样率 `sr` 能如实表示的最高频率是 `sr/2`（**Nyquist 频率**）。超过它，高频会折叠成一个假的低频——这就是**混叠**。

**为什么会折叠**：采样只记录离散时刻的振幅，丢失了两次采样之间的连续细节。超过 Nyquist 的频率 `f` 与其镜像频率在每个采样点上产生完全相同的数值，因此两者在离散域无法区分。`round(f/sr)` 找到最近的整数倍采样率（即最近的折叠中心 `k·sr`），`abs(f - k·sr)` 计算真实频率到这个折叠中心的距离，得到视在频率。例如：`sr=8000, f=6000` → 折叠中心 `1×8000=8000`，距离 `|6000-8000|=2000 Hz`，6 kHz 听起来像 2 kHz。

## 实验入口：把声音拆成可观察的数组

用 `sr=8000` 和极短时长（0.01 s）生成采样点，让数组足够小，可以直接打印。后续代码会对比 `true_freq=6000` 的高频波和折叠后的低频波，验证两者在离散采样点上完全重合。

In [ ]:
import numpy as np
from aurora.audio import sine
print('就绪')

## 动手观察：序列怎样一步步变成信号

修改 `sample_rate`、`duration`、`freq`，观察打印出的时间轴长度（`N`）和采样点间距（`1/sr`）如何随之变化。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：把所有频率都折回 Nyquist 范围

这个实验遍历 0–20 Hz 的频率，用 `|f - sr·round(f/sr)|` 计算每个频率在 `sr=10` 时的视在频率，展示 Nyquist 两侧的频率如何对称折叠。

In [ ]:
import numpy as np

sample_rate = 10
nyquist = sample_rate / 2
freqs = np.arange(0, 21)

for f in freqs:
    alias = abs(((f + nyquist) % sample_rate) - nyquist)
    print(f'true={f:2d}Hz -> alias={alias:4.1f}Hz')


## 2. ✏️ 你的任务：预测折叠后的视在频率

折叠公式：`alias = |freq - sr · round(freq/sr)|`

**推理路线**：
1. 采样率 `sr` 把频率轴切成宽度为 `sr` 的周期性区间；`round(freq/sr)` 找到 `freq` 所在区间的最近整数倍 `k`，即最近的折叠中心 `k·sr`。
2. `freq - k·sr` 是 `freq` 到该中心的带符号距离，取绝对值得到视在频率。
3. 结果必然落在 `[0, sr/2]`：Nyquist 左侧和右侧等距的两个频率折叠到同一正值，这就是对称折叠。

**参考输入输出**：`sr=8000, freq=6000` → Nyquist=4000；k=round(6000/8000)=1；alias=|6000-8000|=2000 Hz（听起来像 2 kHz）

<details><summary>点击查看参考实现</summary>

```python
def predict_alias_freq(freq, sample_rate):
    k = round(freq / sample_rate)
    return abs(freq - k * sample_rate)
```

</details>

### 写代码前，先把变量表补完整

写 `predict_alias_freq` 前明确三件事：
- 输入：`freq`（待测频率，Hz）和 `sample_rate`（采样率，Hz）
- 关键步骤：计算 `k = round(freq / sample_rate)`，再求 `abs(freq - k * sample_rate)`
- 返回：float，即该频率的视在频率（范围 0 到 `sample_rate / 2`）

In [ ]:
def predict_alias_freq(freq, sample_rate):
    # ✏️ TODO: 返回混叠后的视在频率
    return None  # ← 改这里


## 3. 实验：用 8kHz 采 6kHz（6kHz 高于 Nyquist 4kHz，必混叠）

`sr=8000` 的 Nyquist 是 4000 Hz。`freq=6000` 超过 Nyquist，`predict_alias_freq(6000, 8000)` 应返回 2000 Hz。下面代码生成两段信号：真实的 6 kHz 采样波和预测的 2 kHz 纯音，打印两者在同一组采样点上的差值——差值接近 0 即证明混叠成立。

In [ ]:
sr = 8000          # Nyquist = 4000 Hz
true_freq = 6000   # 高于 Nyquist
alias = predict_alias_freq(true_freq, sr)
print('预测视在频率:', alias, 'Hz')
assert 0 <= alias <= sr/2, '视在频率应落在 0..Nyquist'

x_aliased = sine(true_freq, duration=0.01, sample_rate=sr)
x_lowfreq = sine(alias,     duration=0.01, sample_rate=sr)
diff = float(np.max(np.abs(x_aliased - x_lowfreq)))
print('采样点与低频版差异:', f'{diff:.2e}', '(越接近0越说明混叠成立)')

## 4. 画图：高频采样点 vs 那个假低频，几乎重合

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(x_aliased, 'o-', ms=4, label=f'sampled {true_freq} Hz @ {sr} Hz')
plt.plot(x_lowfreq, 'x--', ms=5, label=f'pure {alias:.0f} Hz')
plt.title('Aliasing: a 6 kHz tone masquerades as a low tone')
plt.xlabel('sample n'); plt.legend(); plt.tight_layout(); plt.show()

**结论**：欠采样让高频伪装成低频。这就是录音前要"抗混叠滤波"的原因。

**🎉 完成后**：`git commit -m 'learn: Day 3 aliasing'`

## 🎨 图示：混叠——6kHz 被 8kHz 采样后伪装成低频

In [ ]:
import aviz; aviz.style()
aviz.aliasing(6000, 8000);

In [ ]:
sr = 12
nyq = sr / 2
for f in [1, 5, 6, 7, 11, 13, 17]:
    alias = abs(((f + nyq) % sr) - nyq)
    status = '安全' if f <= nyq else '伪装'
    print(f'{f:>2}Hz -> {alias:>4.1f}Hz  {status}')


## 参数实验：扫描 0–32000 Hz，观察折叠规律

固定 `sr=16000`，把 `freq` 从 0 扫到 32000，每 1000 Hz 打印 `predict_alias_freq`，观察结果在 0–8000 Hz 范围内折叠：16000 Hz 折回 0 Hz，24000 Hz 折回 8000 Hz，32000 Hz 再次折回 0 Hz。把下面代码粘贴到空白单元格运行：

```python
sr = 16000
for f in range(0, 33000, 1000):
    alias = predict_alias_freq(f, sr)
    print(f'{f:>6} Hz  ->  {alias:>6.0f} Hz')
```

规律：折叠图案每隔 `sr` Hz 重复一次；Nyquist（8000 Hz）是对称轴，9000 Hz 和 7000 Hz 产生相同的视在频率（都是 1000 Hz）。

In [ ]:
sr = 12
nyq = sr / 2
for f in [1, 5, 6, 7, 11, 13, 17]:
    alias = abs(((f + nyq) % sr) - nyq)
    status = '安全' if f <= nyq else '伪装'
    print(f'{f:>2}Hz -> {alias:>4.1f}Hz  {status}')


## 本课收束

现在可以用 `predict_alias_freq(freq, sr)` 把任意频率映射到 0–Nyquist 范围内的视在频率。画图验证会显示：6 kHz 高频的采样点与 2 kHz 低频完全重合，在离散域无法区分。Aurora 音频管道在 16 kHz 采样率下只提取 0–8 kHz 的 mel 特征；8 kHz 以上的能量若未被抗混叠滤波器截断，会以假低频的形式污染模型输入。下一节（窗函数）在同样的采样点数组上加汉宁窗，压制频谱泄漏，为 FFT 做准备。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))
